# **Modeling Evaluation #1 (Model-1): Ranking Quality, Fairness & Guardrail Adherence (Lead: Rajesh Mattaparthi)**
This is Rajesh Mattaparthi from Group13; I've taken the model evals work in my group - team A;
This note book is to run the evals for the model1 that was created by Atanu 



Given single lendar's list of recomondations sorted in the rank, we have to find the relavance;
The idea is to create 2 functions to get the relevance:  
**Discounted Cumulative Gain (DCG@k):**  
**Normalized DCG@k:**

In [1]:
#import the required libraries
import pandas as pd
import numpy as np
import math

In [2]:
def dcg_k( relevances, k=10):
    # take the first k values from relevances
    relvs = relevances[:k]
    total =0.0 
    for i , rel in enumerate (relvs, start=1):
        discount = math.log2(i+1)
        total += rel/discount
    return total

In [3]:
# Lets test the above function if it's working or not
scores = [3, 2, 3, 0, 1]
print(dcg_k(scores, 3))   


5.7618595071429155


In [4]:
def ndcg_k(relevances, k=10):
    # we first need to find the actual DCG for the given order
    actual_val = dcg_k(relevances, k)

    ideal_order = sorted(relevances , reverse = True)
    ideal_val = dcg_k(ideal_order, k)

    if ideal_val == 0:
        return 0.0

    return actual_val/ideal_val
    
    

In [5]:
#let's test this function
scores = [3, 2, 3, 0, 1]
print(ndcg_k(scores, 3))


0.9777813616305049


**Let's compute the mean NDCG**

In [6]:
def compute_mean_ndcg(df_predictions, k, query_col, rank_col, label_col, target=0.8):
    scores_per_query = []

    for lender_id, group in df_predictions.groupby(query_col):
        sorted_group = group.sort_values(rank_col)
        relevances = sorted_group[label_col].tolist()
        score = ndcg_k(relevances, k)
        scores_per_query.append(score)

    if scores_per_query:
        mean_score = sum(scores_per_query) / len(scores_per_query)
    else:
        mean_score = 0.0

    return {
        f"ndcg_at_{k}_mean": mean_score,
        f"ndcg_at_{k}_per_query": scores_per_query,
        "n_queries": len(scores_per_query),
        f"meets_target_{target}": mean_score >= target
    }

In [7]:
# # let's test the above function
# result = compute_mean_ndcg(
#     df_predictions,
#     k=10,
#     query_col="lender_serial_number",   
#     rank_col="final_rank_position",
#     label_col="relevance"              
# )


**Let's compute the selection rate variance**

In [8]:
def compute_selection_rate_var(df_predictions, k, ranked_col="final_rank_position", protected_col=("borrower_gender", "region")):
    #let's pick top k rows
    top_k_rows = df_predictions[df_predictions[ranked_col] <= k]

    #Check if the top k rows has some values, then only run
    if len(top_k_rows) == 0:
        result = {}
        for col in protected_col:
            result[col] = {
                "selection_rates": {}, 
                "selection_rate_range": 0.0,
                "meets_target_le_5pct": True
            }
        return result
    
    results = {}
    for val in protected_col:
        if val not in top_k_rows.columns:
            results[val] = {"error" : "Column not found"}
            continue

        #get the count of frequencies
        group_count = top_k_rows[val].value_counts(dropna=False)
        total = group_count.sum()
        
        if total == 0.0:
            rates = {}
        else:
            rates = (group_count/total).to_dict()

        if len(rates)>1:
            selection_rate_range = max(rates.values()) - min(rates.values())
        else:
            selection_rate_range = 0.0

        results[val] = {
            "selection_rates": rates,
            "selection_rate_range": selection_rate_range,
            "meets_target_le_5pct": selection_rate_range <= 0.05
        } 
    return results

**Let's do the evaluation for the guard rails**

In [9]:
def compute_guardrail_adherence(df_guardrail, voilation_col="violation_flag"):
    #check if the guardrail dataframe is empty or not
    if len(df_guardrail) == 0: 
        return {
            "no_of_checks": 0,
            "count_of_voilations": 0,
            "guardrail_voilation_rate": None,
            "guardrail_adherence_rate": None,
            "meets_target_0pct_violations": None
        }
    total_checks = len(df_guardrail)     
    count_of_voilations = df_guardrail[voilation_col].sum()

    #let's compute the adherence rate
    guardrail_voilation_rate = count_of_voilations/total_checks
    adherence_rate = 1- guardrail_voilation_rate

    #return the result
    return {
        "no_of_checks": total_checks,
        "count_of_voilations":count_of_voilations,
        "guardrail_voilation_rate":guardrail_voilation_rate,
        "guardrail_adherence_rate": adherence_rate,
        "meets_target_0pct_violations": count_of_voilations == 0
        
    }
    

In [10]:
#This function does the final evaluation by invoking previously built functions 
# with actual model predictions as inputs
def run_full_evaluation(predictions_df: pd.DataFrame, guardrail_df: pd.DataFrame, k: int) -> dict:

    # below is our target scores
    NDCG_TARGET = 0.80
    FAIRNESS_TARGET = 0.05
    GUARDRAIL_TARGET = 1.0

    # compute_mean_ndcg / compute_selection_rate_var both need a rank column,
    # but the source CSV only has raw pred_score - derive it here (highest
    # score = rank 1) without touching those function definitions.
    predictions_df = predictions_df.copy()

    n_missing_scores = predictions_df["pred_score"].isna().sum()
    if n_missing_scores > 0:
        print(f"Warning: dropping {n_missing_scores} row(s) with missing pred_score before ranking.")
        predictions_df = predictions_df.dropna(subset=["pred_score"])

    # sort the values - explicit, deterministic sort before ranking.
    # Sorting first makes the tie-break intentional and
    # reproducible instead of an incidental side  effect of file order. 
    predictions_df = predictions_df.sort_values(
        ["lender_serial_number", "pred_score"], ascending=[True, False], kind="stable"
    )
    predictions_df["final_rank_position"] = (
        predictions_df.groupby("lender_serial_number")["pred_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    ndcg_result = compute_mean_ndcg(
        predictions_df, k,
        query_col="lender_serial_number",
        rank_col="final_rank_position",
        label_col="relevance",
        target=NDCG_TARGET,
    )
    fairness_result = compute_selection_rate_var(
        predictions_df, k,
        ranked_col="final_rank_position",
        protected_col=("rural_tier",),
    )
    guardrail_result = compute_guardrail_adherence(
        guardrail_df, voilation_col="violation_triggered"
    )

    print(f"--- NDCG@{k} ---")
    #  reference the dynamic key 
    ndcg_mean = ndcg_result[f"ndcg_at_{k}_mean"]
    ndcg_ok = ndcg_mean >= NDCG_TARGET
    print(f"mean NDCG@{k}: {ndcg_mean:.4f}  "
          f"(target >= {NDCG_TARGET:.2f})  -> {'PASS' if ndcg_ok else 'FAIL'}")
    if ndcg_mean == 1.0:
        # a perfect NDCG across every query is unusual for a
        # real model - flagging it so it gets a second look (possible label
        # leakage, e.g. "relevance" derived from the same signal as pred_score).
        print(f"Note: NDCG@{k} is a perfect 1.0000 - double check that 'relevance' "
              "is independent ground truth and not derived from pred_score/rank.")

    print(f"\n--- fairness (selection-rate range) ---")
    for attribute, info in fairness_result.items():
        # handle the "Column not found" case instead of
        # letting info["selection_rate_range"] raise a KeyError.
        if "error" in info:
            print(f"{attribute}: {info['error']} - skipping fairness check")
            continue
        selection_rate_range = info["selection_rate_range"]
        fairness_ok = selection_rate_range <= FAIRNESS_TARGET
        print(f"{attribute}: selection rate range = {selection_rate_range:.4f}  "
              f"(target <= {FAIRNESS_TARGET:.2%})  -> {'PASS' if fairness_ok else 'FAIL'}")

    print(f"\n--- guardrail adherence ---")
    guardrail_rate = guardrail_result["guardrail_adherence_rate"]
    # guardrail_adherence_rate is None when the guardrail
    # log is empty; comparing None >= GUARDRAIL_TARGET used to raise a
    # TypeError here.
    if guardrail_rate is None:
        guardrail_ok = None
        print("adherence rate: N/A (no guardrail checks in guardrail_df)  -> SKIPPED")
    else:
        guardrail_ok = guardrail_rate >= GUARDRAIL_TARGET
        print(f"adherence rate: {guardrail_rate:.2%}  "
              f"(target = {GUARDRAIL_TARGET:.0%})  -> {'PASS' if guardrail_ok else 'FAIL'}")
        if guardrail_rate < 0.5:
            # an adherence rate this low is extreme - flagging it in case
            #  "violation_triggered" semantics are inverted in the source data. 
            print("Note: guardrail adherence is very low - confirm 'violation_triggered' "
                  "is True on violations, not on passed checks.")

    return {
        "ndcg": ndcg_result,
        "fairness": fairness_result,
        "guardrail": guardrail_result,
    }


In [11]:
# Now it's time to run the evals for the actual model output;
# These are the predictions shared by Team A - Model 1;
predictions_df = pd.read_csv("model1_predictions 1.csv")
guardrail_df = pd.read_csv("model1_guardrail_log.csv")
print("The final results are as follows:")
print("********************************")
result = run_full_evaluation(predictions_df, guardrail_df, k=10)

The final results are as follows:
********************************
--- NDCG@10 ---
mean NDCG@10: 1.0000  (target >= 0.80)  -> PASS
Note: NDCG@10 is a perfect 1.0000 - double check that 'relevance' is independent ground truth and not derived from pred_score/rank.

--- fairness (selection-rate range) ---
rural_tier: selection rate range = 0.6393  (target <= 5.00%)  -> FAIL

--- guardrail adherence ---
adherence rate: 2.09%  (target = 100%)  -> FAIL
Note: guardrail adherence is very low - confirm 'violation_triggered' is True on violations, not on passed checks.


In [13]:
#This is iteration 2 testing
# These are the predictions shared by Team A - Model 1;
predictions_df = pd.read_csv("model1_predictions_updated.csv")
guardrail_df = pd.read_csv("model1_guardrail_log_updated.csv")
print("The final results are as follows: iteration 2")
print("********************************")
result = run_full_evaluation(predictions_df, guardrail_df, k=10)

The final results are as follows: iteration 2
********************************
--- NDCG@10 ---
mean NDCG@10: 1.0000  (target >= 0.80)  -> PASS
Note: NDCG@10 is a perfect 1.0000 - double check that 'relevance' is independent ground truth and not derived from pred_score/rank.

--- fairness (selection-rate range) ---
rural_tier: selection rate range = 0.6413  (target <= 5.00%)  -> FAIL

--- guardrail adherence ---
adherence rate: 10.47%  (target = 100%)  -> FAIL
Note: guardrail adherence is very low - confirm 'violation_triggered' is True on violations, not on passed checks.


**AI Usage Memo**

Used the Copilot for the code issues and review, find the link here - https://copilot.microsoft.com/shares/tf9fVQxDqaN2jE3gxrjGF 

Used Claude Code to debug the issues during the testing and code beautification. Couldn't capture the logs as it doesn't have the option to share debugging log links